# P0 — pykrx 데이터 깊이 프로브 (KRX 로그인 필요 · OPEN API 키 불필요)

**목적:** 한국 가격·시총·PBR이 각각 **몇 년까지** 닿는지 확인한다.
이게 표본 기간을 좌우한다 — 특히 PBR이 2015년 이전으로 닿으면, OpenDART(재무제표 2015+) 없이도
B/M(가치 팩터)을 더 길게 뽑는 우회로가 생긴다.

> **2025-12-27 변경:** KRX 정보데이터시스템이 회원제 'KRX Data Marketplace'로 전환되면서 **로그인이 필수**가 됐다(데이터 조회 자체는 무료).
> 시총·PBR·전종목 펜더멘털은 로그인 없이 빈 응답을 반환하며, pykrx 내부에서 JSON 파싱 에러로 떨어진다.
> 아래 **0) 설정** 셀을 먼저 끝내고, **커널을 재시작한 뒤** 위에서부터 실행한다.
>
> 결과를 본 다음, 각 칸의 **가장 이른 날짜와 종목 수**를 알려줘. 그걸로 표본 기간을 확정한다.

## 0) KRX 회원가입 + .env 설정 (최초 1회)

**가입 (무료):**
1. https://data.krx.co.kr → 우측 상단 **회원가입**
2. 개인 회원 → 본인인증 또는 소셜 로그인
3. 가입한 ID/PW가 곳 `KRX_ID` / `KRX_PW`

> 참고: pykrx가 쓰는 건 이 **일반 회원 계정**이다. `openapi.krx.co.kr`의 공식 OPEN API 인증키와는 다르며, 인증키 신청은 필요 없다.

**`.env` 파일 (프로젝트 루트 `d:\ff-replication\.env`):**
```
KRX_ID=내_krx_아이디
KRX_PW=내_krx_비밀번호
```

**추가 설정:**
- `pip install python-dotenv`
- `.gitignore`에 `.env` 한 줄 추가 (자격증명이 git에 올라가지 않도록)
- **설정 후 커널을 재시작하고 먼 위에서부터 실행** — pykrx는 import 시점에 로그인하므로, 한번 import된 뒤엔 .env를 다시 읽어도 재로그인하지 않는다.

## 준비

`pip install pykrx pandas python-dotenv`

In [1]:
# 1) KRX 자격증명을 pykrx import보다 "먼저" 로드한다.
#    pykrx는 import 시점에 KRX 로그인을 시도하므로, 이 순서가 중요하다.
import os
from dotenv import load_dotenv

load_dotenv('d:/ff-replication/.env')  # 프로젝트 루트의 .env에서 KRX_ID, KRX_PW를 읽는다
assert os.getenv('KRX_ID') and os.getenv('KRX_PW'), (
    'KRX_ID / KRX_PW가 설정되지 않았다. 프로젝트 루트에 .env를 만들고 값을 채운 뒤 '
    '커널을 재시작하고 처음부터 실행한다.'
)

# 2) 자격증명이 환경에 있는 상태에서 import → 로그인 성공
from pykrx import stock
import pandas as pd

TODAY = pd.Timestamp.today().strftime('%Y%m%d')
SAMSUNG = '005930'   # 삼성전자 (1975년 상장 — 깊은 역사 기준 종목)
print('로그인 계정:', os.getenv('KRX_ID')[:3] + '***')
print('오늘:', TODAY)

KRX 로그인 시도...
  로그인 ID: taeyoon3684
KRX 로그인 완료.
  로그인 시간: 2026-06-30 23:38:41
  만료 시간: 2026-07-01 00:38:41
로그인 계정: tae***
오늘: 20260630


## 1) 가격(OHLCV) 깊이

삼성전자 일별 종가를 1995년부터 요청 → 실제로 데이터가 시작되는 날짜를 본다.

> 주의: OHLCV는 Naver 소스라 로그인 없이도 돌아가지만, 과거 ~3000행(≈12년)에서 캐핑될 수 있다.
> 시작 날짜가 2014년근처에서 멈추면 그건 소스 한계이므로, 더 긴 가격은 로그인 후 `get_market_cap`(KRX 소스, 종가 포함) 또는 FinanceDataReader로 따로 받는다.

In [2]:
px = stock.get_market_ohlcv_by_date('19950101', TODAY, SAMSUNG)
print('가격 데이터:', px.index.min(), '~', px.index.max(), '/', len(px), '행')
px.head(2)

가격 데이터: 2014-04-07 00:00:00 ~ 2026-06-30 00:00:00 / 3000 행


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2014-04-07,27940,27940,27480,27940,215235,NaN
2014-04-08,27740,27980,27500,27880,212164,-0.214746


## 2) 시가총액 깊이 (사이즈 팩터용) ← 로그인 필요

In [3]:
cap = stock.get_market_cap_by_date('19950101', TODAY, SAMSUNG)
print('시총 데이터:', cap.index.min(), '~', cap.index.max(), '/', len(cap), '행')
cap.head(2)

시총 데이터: 1995-05-02 00:00:00 ~ 2026-06-30 00:00:00 / 7852 행


,시가총액,거래량,거래대금,상장주식수
날짜,,,,
1995-05-02,6497053077500,139560,16676735000,54368645
1995-05-03,6714527657500,382980,47649710000,54368645


## 3) 펜더멘털(PBR·BPS) 깊이 ← B/M 우회로의 핵심 (로그인 필요)

KRX가 직접 공시하는 PBR이 몇 년부터 유효한지. 여기가 2015년 이전으로 닿으면 표본이 길어진다.

In [4]:
fun = stock.get_market_fundamental_by_date('19950101', TODAY, SAMSUNG)
print('펜더멘털 컬럼:', list(fun.columns))
print('전체 범위:', fun.index.min(), '~', fun.index.max())
if 'PBR' in fun.columns:
    valid = fun[fun['PBR'] > 0]
    print('PBR 유효 시작:', valid.index.min() if len(valid) else '없음')
else:
    print('PBR 컬럼 없음 — 로그인 상태 또는 함수 반환 형태를 확인할 것')
fun.head(2)

펜더멘털 컬럼: ['BPS', 'PER', 'PBR', 'EPS', 'DIV', 'DPS']
전체 범위: 1995-05-02 00:00:00 ~ 2026-06-30 00:00:00
PBR 유효 시작: 2002-04-23 00:00:00


,BPS,PER,PBR,EPS,DIV,DPS
날짜,,,,,,
1995-05-02,0,0.0,0.0,0,0.0,0
1995-05-03,0,0.0,0.0,0,0.0,0


## 4) 과거 유니버스(종목 수) — KOSPI / KOSDAQ

여러 과거 날짜에서 종목 리스트가 제대로 나오는지 (유니버스 구성이 그 시점에 가능한지).

In [5]:
for d in ['20001229', '20051230', '20101230', '20151230', '20201230']:
    try:
        k = len(stock.get_market_ticker_list(d, market='KOSPI'))
        q = len(stock.get_market_ticker_list(d, market='KOSDAQ'))
        print(f'{d}:  KOSPI {k}개,  KOSDAQ {q}개')
    except Exception as e:
        print(f'{d}:  실패 — {type(e).__name__}: {e}')

20001229:  KOSPI 902개,  KOSDAQ 615개
20051230:  KOSPI 858개,  KOSDAQ 931개
20101230:  KOSPI 927개,  KOSDAQ 1035개
20151230:  KOSPI 887개,  KOSDAQ 1154개
20201230:  KOSPI 917개,  KOSDAQ 1471개


## 5) 특정 날짜 전체 시장 PBR (B/M 정렬이 가능한가) ← 로그인 필요

한 날짜에 시장 전체의 PBR을 한 번에 받을 수 있어야 25개 바구니로 정렬할 수 있다.

In [6]:
for d in ['20151230', '20101230', '20051230']:
    try:
        f = stock.get_market_fundamental_by_ticker(d, market='KOSPI')
        nvalid = int((f['PBR'] > 0).sum()) if 'PBR' in f.columns else 0
        print(f'{d} KOSPI:  {len(f)}종목 중 PBR 유효 {nvalid}개')
    except Exception as e:
        print(f'{d}:  실패 — {type(e).__name__}: {e}')

20151230 KOSPI:  854종목 중 PBR 유효 729개
20101230 KOSPI:  872종목 중 PBR 유효 702개
20051230 KOSPI:  819종목 중 PBR 유효 0개
